# 01 — Binance Market Data Exploration

## Goal

Understand the structure of Binance market data before building the data pipeline.

Tasks:

- Inspect Binance API structure
- Understand trade messages
- Understand order book depth updates
- Examine timestamps and ordering
- Identify fields needed for limit order book reconstruction

This notebook is **exploratory**.  
Production code will later move into:



---

# 1. Exchange and Market Selection

For this project we use:

Exchange: Binance
Pair: BTCUSDT

- High liquidity
- Access to limit order book
- Good api support
- Free to use




# Load Required Libraries

In [223]:
import pandas as pd
import numpy as np
from binance.client import Client
import time
import black

# Get Order Book
I want to just get data one time to allow for reproducible results


In [224]:
client = Client()

Check latency, since binance doesnt give time with the non websocket depth snapshots

In [225]:
t0 = int(time.time() * 1000)

depth = client.get_order_book(symbol="BTCUSDT")

t1 = int(time.time() * 1000)

rtt = t1 - t0
latency = rtt / 2
latency

119.0

Latency ~ 151 ms so using 1 second intervals for snapshots should in general be fine (missing intra second data) - future update is to add in websocket intrasecond info

In [226]:
depth_samples = []

for i in range(5):
    timestamp = int(time.time() * 1000)
    depth = client.get_order_book(symbol="BTCUSDT", limit=100)
    bids = depth["bids"]
    asks = depth["asks"]
    depth_samples.append(
        {
            "timestamp": timestamp,
            "lastUpdateId": depth["lastUpdateId"],
            "bids": depth["bids"],
            "asks": depth["asks"],
        }
    )
    time.sleep(1)
trades = client.get_recent_trades(symbol="BTCUSDT", limit=1000)

depth_samples



[{'timestamp': 1773047452780,
  'lastUpdateId': 89730922218,
  'bids': [['67621.99000000', '1.18578000'],
   ['67621.98000000', '0.00008000'],
   ['67621.82000000', '0.13316000'],
   ['67621.68000000', '0.00009000'],
   ['67620.00000000', '0.00008000'],
   ['67619.97000000', '0.00008000'],
   ['67619.88000000', '0.00009000'],
   ['67619.25000000', '0.00008000'],
   ['67619.24000000', '0.05153000'],
   ['67619.22000000', '0.00008000'],
   ['67619.21000000', '0.02737000'],
   ['67619.20000000', '0.00625000'],
   ['67618.93000000', '0.00236000'],
   ['67618.77000000', '0.03044000'],
   ['67618.20000000', '0.00008000'],
   ['67617.50000000', '0.00200000'],
   ['67617.46000000', '0.00012000'],
   ['67617.34000000', '0.00009000'],
   ['67617.12000000', '0.00008000'],
   ['67617.00000000', '0.00009000'],
   ['67616.98000000', '0.00022000'],
   ['67616.80000000', '0.00008000'],
   ['67616.64000000', '0.00017000'],
   ['67616.21000000', '0.00139000'],
   ['67615.97000000', '0.00014000'],
   ['6

Create bids df exploded in bids same with asks

In [227]:
bids_df = pd.DataFrame(depth_samples)
bids_df = bids_df[["timestamp", "lastUpdateId", "bids"]].explode("bids")

bids_df["bids_price"], bids_df["bids_volume"] = zip(*bids_df["bids"])

bids_df["bids_price"] = bids_df["bids_price"].astype(float)
bids_df["bids_volume"] = bids_df["bids_volume"].astype(float)

bids_df = bids_df.drop(columns="bids")
bids_df.head()

,timestamp,lastUpdateId,bids_price,bids_volume
0,1773047452780,89730922218,67621.990000,1.185780
0,1773047452780,89730922218,67621.980000,0.000080
0,1773047452780,89730922218,67621.820000,0.133160
0,1773047452780,89730922218,67621.680000,0.000090
0,1773047452780,89730922218,67620.000000,0.000080


In [228]:
asks_df = pd.DataFrame(depth_samples)
asks_df = asks_df[["timestamp", "lastUpdateId", "asks"]].explode("asks")
asks_df["asks_price"], asks_df["asks_volume"] = zip(*asks_df["asks"])

asks_df["asks_price"] = asks_df["asks_price"].astype(float)
asks_df["asks_volume"] = asks_df["asks_volume"].astype(float)

asks_df = asks_df.drop(columns="asks")
asks_df.head()

,timestamp,lastUpdateId,asks_price,asks_volume
0,1773047452780,89730922218,67622.000000,0.916060
0,1773047452780,89730922218,67622.010000,0.000160
0,1773047452780,89730922218,67622.750000,0.000080
0,1773047452780,89730922218,67623.360000,0.000080
0,1773047452780,89730922218,67624.190000,0.000160


now for both asks and bids gets the highest price for each timestamp

In [229]:
max_bid = bids_df.groupby("timestamp")["bids_price"].max().rename("max_bids")
max_ask = asks_df.groupby("timestamp")["asks_price"].max().rename("max_asks")
max_bid_vol = bids_df.groupby("timestamp")["bids_volume"].max().rename("max_bids_vol")
max_ask_vol = asks_df.groupby("timestamp")["asks_volume"].max().rename("max_asks_vol")

In [230]:
combined_df = pd.concat([max_bid, max_ask, max_bid_vol, max_ask_vol], axis=1)
combined_df.reset_index(inplace=True)
combined_df

,timestamp,max_bids,max_asks,max_bids_vol,max_asks_vol
0,1773047452780,67621.990000,67642.000000,2.790720,1.995570
1,1773047454017,67621.990000,67640.160000,1.997570,2.025570
2,1773047455255,67621.990000,67640.000000,1.997570,2.025570
3,1773047456488,67621.990000,67639.220000,1.997570,2.025570
4,1773047457723,67619.200000,67638.640000,1.997570,2.450640


Create features

- timestamp
- best_bid
- best_ask
- mid_price
- spread
- bid_volume
- ask_volume
- imbalance
- microprice
  - to do weighted microprice (maybe add in 5 layers to lob not just best price)
- trade_imbalance



In [231]:
combined_df["mid_price"] = (combined_df["max_bids"] + combined_df["max_asks"]) / 2
combined_df["spread"] = combined_df["max_asks"] - combined_df["max_bids"]

combined_df["imbalance"] = (
    combined_df["max_bids_vol"] - combined_df["max_asks_vol"]
) / (combined_df["max_bids_vol"] + combined_df["max_asks_vol"])


combined_df["liquidity"] = combined_df["max_bids_vol"] + combined_df["max_asks_vol"]
combined_df["microprice"] = (combined_df["max_bids_vol"]*combined_df["max_bids"] + combined_df["max_asks_vol"]*combined_df["max_asks"]) / combined_df["liquidity"]

combined_df

,timestamp,max_bids,max_asks,max_bids_vol,max_asks_vol,mid_price,spread,imbalance,liquidity,microprice
0,1773047452780,67621.990000,67642.000000,2.790720,1.995570,67631.995000,20.010000,0.166131,4.786290,67630.332862
1,1773047454017,67621.990000,67640.160000,1.997570,2.025570,67631.075000,18.170000,-0.006960,4.023140,67631.138229
2,1773047455255,67621.990000,67640.000000,1.997570,2.025570,67630.995000,18.010000,-0.006960,4.023140,67631.057672
3,1773047456488,67621.990000,67639.220000,1.997570,2.025570,67630.605000,17.230000,-0.006960,4.023140,67630.664958
4,1773047457723,67619.200000,67638.640000,1.997570,2.450640,67628.920000,19.440000,-0.101854,4.448210,67629.910025


trade df

In [232]:
trades_df = pd.DataFrame(trades)
trades_df = trades_df.astype({
    "price": float,
    "qty": float,
    "quoteQty": float,
    "time": "int64",
    "isBuyerMaker": bool,
    "isBestMatch": bool,
})
trades_df

,id,price,qty,quoteQty,time,isBuyerMaker,isBestMatch
0,6082553701,67629.760000,0.002360,159.606234,1773047436395,True,True
1,6082553702,67629.760000,0.000080,5.410381,1773047436395,True,True
2,6082553703,67629.760000,0.000080,5.410381,1773047436395,True,True
3,6082553704,67629.760000,0.000080,5.410381,1773047436395,True,True
4,6082553705,67629.760000,0.000080,5.410381,1773047436395,True,True
...,...,...,...,...,...,...,...
995,6082554696,67619.200000,0.000160,10.819072,1773047459575,True,True
996,6082554697,67619.200000,0.000690,46.657248,1773047459575,True,True
997,6082554698,67619.200000,0.000080,5.409536,1773047459575,True,True
998,6082554699,67619.200000,0.000080,5.409536,1773047459575,True,True


I need to segment trades info into 5 time slots each for each second (need to add in buffer for latency)

In [233]:
pd.set_option("display.float_format", "{:.6f}".format)
trades_df_interval_masks = []

timestamps = combined_df["timestamp"].values
intervals = zip(timestamps[:-1],timestamps[1:])

for t0, t1 in intervals:
    mask = (trades_df["time"]>t0) & (trades_df["time"]<t1)
    trades_df_interval_masks.append(mask) 

    

Check to see if trades correct time intervals

In [234]:
trade_slice_1= trades_df[trades_df_interval_masks[0]]
        
trade_slice_1["time"].describe()

count              13.000000
mean    1773047453762.923096
std               357.401143
min     1773047453136.000000
25%     1773047453951.000000
50%     1773047453951.000000
75%     1773047453951.000000
max     1773047453951.000000
Name: time, dtype: float64

In [235]:
trade_slice_1

,id,price,qty,quoteQty,time,isBuyerMaker,isBestMatch
924,6082554625,67621.990000,0.000080,5.409759,1773047453136,True,True
925,6082554626,67621.990000,0.000080,5.409759,1773047453136,True,True
926,6082554627,67621.990000,0.000950,64.240891,1773047453136,True,True
927,6082554628,67622.000000,0.000160,10.819520,1773047453951,False,True
928,6082554629,67622.000000,0.000160,10.819520,1773047453951,False,True
929,6082554630,67622.000000,0.000160,10.819520,1773047453951,False,True
930,6082554631,67622.000000,0.000690,46.659180,1773047453951,False,True
931,6082554632,67622.000000,0.000080,5.409760,1773047453951,False,True
932,6082554633,67622.000000,0.000160,10.819520,1773047453951,False,True
933,6082554634,67622.000000,0.000080,5.409760,1773047453951,False,True


In [236]:
for i, mask in enumerate(trades_df_interval_masks):
    print(i, mask.sum())

0 13
1 3
2 1
3 20


get trade features do for 1 slice then create function to do for all slices
- trade_count
- buy_count
- sell_count
- total_trade_volume
- buy_volume
- sell_volume
- trade_imbalance
- trade_flow_ratio
- avg_trade_size
- max_trade_size
- std_trade_size
- vwap


In [ ]:
trade_count = len(trade_slice_1)
buy_count = len(trade_slice_1[ trade_slice_1["isBuyerMaker"]==True])
sell_count = len(trade_slice_1[ trade_slice_1["isBuyerMaker"]==False])

total_trade_volume = trade_slice_1["qty"].sum()

buy_volume = trade_slice_1.loc[trade_slice_1["isBuyerMaker"]==True,"qty"].sum()
sell_volume = trade_slice_1.loc[trade_slice_1["isBuyerMaker"]==False,"qty"].sum()

trade_imbalance = (buy_volume - sell_volume) / total_trade_volume

trade_flow_ratio = buy_volume/sell_volume

avg_trade_size = total_trade_volume / trade_count

max_trade_size = trade_slice_1["qty"].max()
min_trade_size = trade_slice_1["qty"].min()

std_trade_size = trade_slice_1["qty"].std()

vwap = (trade_slice_1["price"]*trade_slice_1["qty"]).sum() / total_trade_volume


np.float64(67621.9997961058)